In [2]:
import torch
import torchvision
from torch import nn
from torch.utils.data import DataLoader

# 1. 准备真实的 CIFAR-10 数据管道
# CIFAR-10 图片分辨率是 32x32 像素，彩色 3 通道
dataset = torchvision.datasets.CIFAR10('../dataset', train=False, transform=torchvision.transforms.ToTensor(), download=True)
dataloader = DataLoader(dataset, batch_size=64, drop_last=True) # 开启 drop_last 保证 batch 规整

# 2. 定义 CIFAR-10 专属的线性网络模型
class CIFAR10_Linear_Model(nn.Module):
    def __init__(self):
        super().__init__()
        # 维度对齐：每个神经元只看单张 CIFAR-10 展平后的 3072 个像素点，输出 10 个分类
        self.linear = nn.Linear(in_features=3072, out_features=10)

    def forward(self, x):
        # 保持第 0 维（Batch_size 64）不动，只把后面的 [3, 32, 32] 展平为 3072
        # 展平后的 x 形状变为: [64, 3072] [2]
        x = torch.flatten(x, start_dim=1)

        # 喂入线性层，输出形状为 [64, 10] (即：同时输出 64 张图片各自的 10 分类得分)
        logits = self.linear(x)
        return logits

# 实例化模型
model = CIFAR10_Linear_Model()

# 测试运行
for data in dataloader:
    imgs, targets = data
    # 原始图像包: [64, 3, 32, 32]
    print("输入图像包的形状:", imgs.shape)

    # 进行前向传播
    outputs = model(imgs)

    # 最终输出: [64, 10] ──> 对应 64 张图各自的预测得分
    print("输出 Logits 的形状:", outputs.shape)
    break

输入图像包的形状: torch.Size([64, 3, 32, 32])
输出 Logits 的形状: torch.Size([64, 10])
